In [46]:
from pathlib import Path
import joblib

import numpy as np
import pandas as pd

import re

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score

In [30]:
ROOT = Path.cwd().parent

DATA_DIR = ROOT / "dataset" / "data"
SPLIT_DIR = ROOT / "dataset" / "splits"
RUNTIME_DIR = ROOT / "dataset" / "runtime"
CHECKPOINT_DIR = ROOT / "checkpoints"

COMBINED_DATA_PATH = DATA_DIR / "combined_train.csv"

TRAIN_SPLIT_PATH = SPLIT_DIR / "train_split.csv"
VALIDATION_SPLIT_PATH = SPLIT_DIR / "validation_split.csv"
TEST_SPLIT_PATH = SPLIT_DIR / "test_split.csv"

TFIDF_VECTORIZER_PATH = (
    CHECKPOINT_DIR / "tfidf_vectorizer.joblib"
)

TFIDF_CLASSIFIER_PATH = (
    CHECKPOINT_DIR / "tfidf_classifier.joblib"
)

ATAE_MODEL_PATH = (
    CHECKPOINT_DIR / "atae_lstm_best.pt"
)

TFIDF_PREDICTIONS_PATH = (
    RUNTIME_DIR / "tfidf_predictions.csv"
)

ATAE_PREDICTIONS_PATH = (
    RUNTIME_DIR / "atae_predictions.csv"
)

COMPARISON_OUTPUT_PATH = (
    RUNTIME_DIR / "model_comparison.csv"
)

EVALUATION_OUTPUT_PATH = (
    RUNTIME_DIR / "held_out_test_predictions.csv"
)

METRICS_OUTPUT_PATH = (
    RUNTIME_DIR / "model_evaluation_metrics.csv"
)

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

labels = [
    "negative",
    "neutral",
    "positive",
    "conflict"
]

LABEL_FIXES = {
    "postive": "positive",
    "po": "positive"
}

print("Project root:", ROOT)
print("Combined labeled data:", COMBINED_DATA_PATH)
print("Saved common test split:", TEST_SPLIT_PATH)
print("TF-IDF model:", TFIDF_CLASSIFIER_PATH)
print("ATAE-LSTM model:", ATAE_MODEL_PATH)

Project root: c:\New folder\Projects\sentiment analysis
Combined labeled data: c:\New folder\Projects\sentiment analysis\dataset\data\combined_train.csv
Saved common test split: c:\New folder\Projects\sentiment analysis\dataset\splits\test_split.csv
TF-IDF model: c:\New folder\Projects\sentiment analysis\checkpoints\tfidf_classifier.joblib
ATAE-LSTM model: c:\New folder\Projects\sentiment analysis\checkpoints\atae_lstm_best.pt


In [3]:
required_paths = [
    TFIDF_PATH,
    ATAE_PATH
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required prediction file(s) missing:\n"
        + "\n".join(str(path) for path in missing_paths)
        + "\n\nRun these notebooks first:\n"
        "1. 04_predict_tfidf.ipynb\n"
        "2. 05_predict_atae_lstm.ipynb"
    )

print("Both prediction files found.")

Both prediction files found.


In [31]:
required_paths = [
    COMBINED_DATA_PATH,
    TFIDF_VECTORIZER_PATH,
    TFIDF_CLASSIFIER_PATH,
    ATAE_MODEL_PATH
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required file(s) missing:\n"
        + "\n".join(str(path) for path in missing_paths)
        + "\n\nMake sure you have run:\n"
        "1. 01_train_tfidf.ipynb\n"
        "2. 02_train_atae_lstm.ipynb"
    )

print("All required training data and model files were found.")

All required training data and model files were found.


In [32]:
def clean_labeled_data(df):
    df = df.copy()

    df["polarity"] = (
        df["polarity"]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace(LABEL_FIXES)
    )

    df = df.dropna(
        subset=[
            "text",
            "aspect_term",
            "polarity"
        ]
    ).copy()

    df["text"] = (
        df["text"]
        .astype(str)
        .str.strip()
    )

    df["aspect_term"] = (
        df["aspect_term"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df = df[
        df["text"].ne("") &
        df["aspect_term"].ne("")
    ].copy()

    df = df[
        df["polarity"].isin(labels)
    ].copy()

    return df.reset_index(drop=True)

In [33]:
labeled_data = pd.read_csv(COMBINED_DATA_PATH)
labeled_data = clean_labeled_data(labeled_data)

print("Usable labeled rows:", len(labeled_data))

print("\nTrue class distribution:")
print(labeled_data["polarity"].value_counts())

display(
    labeled_data[
        [
            "text",
            "aspect_term",
            "polarity"
        ]
    ].head()
)

Usable labeled rows: 2721

True class distribution:
polarity
positive    1193
negative    1013
neutral      470
conflict      45
Name: count, dtype: int64


,text,aspect_term,polarity
0,lenovo personally maybe,lenovo,positive
1,legion like intel chipset great laptop value b...,intel,positive
2,legion like intel chipset great laptop value b...,battery life,positive
3,definitely consider lenovo gaming laptop,lenovo,positive
4,hp user not hp,hp,negative


In [34]:
if (
    TRAIN_SPLIT_PATH.exists()
    and VALIDATION_SPLIT_PATH.exists()
    and TEST_SPLIT_PATH.exists()
):
    train_split = pd.read_csv(TRAIN_SPLIT_PATH)
    validation_split = pd.read_csv(VALIDATION_SPLIT_PATH)
    test_split = pd.read_csv(TEST_SPLIT_PATH)

    print("Loaded existing fixed train/validation/test splits.")

else:
    train_split, temporary_split = train_test_split(
        labeled_data,
        test_size=0.20,
        random_state=42,
        stratify=labeled_data["polarity"]
    )

    validation_split, test_split = train_test_split(
        temporary_split,
        test_size=0.50,
        random_state=42,
        stratify=temporary_split["polarity"]
    )

    train_split = train_split.reset_index(drop=True)
    validation_split = validation_split.reset_index(drop=True)
    test_split = test_split.reset_index(drop=True)

    train_split.to_csv(
        TRAIN_SPLIT_PATH,
        index=False
    )

    validation_split.to_csv(
        VALIDATION_SPLIT_PATH,
        index=False
    )

    test_split.to_csv(
        TEST_SPLIT_PATH,
        index=False
    )

    print("Created and saved fixed common splits.")

Loaded existing fixed train/validation/test splits.


In [35]:
train_split = clean_labeled_data(train_split)
validation_split = clean_labeled_data(validation_split)
test_split = clean_labeled_data(test_split)

print("Train rows:", len(train_split))
print("Validation rows:", len(validation_split))
print("Test rows:", len(test_split))

print("\nHeld-out test class distribution:")
print(test_split["polarity"].value_counts())

Train rows: 2176
Validation rows: 272
Test rows: 273

Held-out test class distribution:
polarity
positive    120
negative    102
neutral      47
conflict      4
Name: count, dtype: int64


In [36]:
tfidf_vectorizer = joblib.load(
    TFIDF_VECTORIZER_PATH
)

tfidf_classifier = joblib.load(
    TFIDF_CLASSIFIER_PATH
)

print("TF-IDF vectorizer:", type(tfidf_vectorizer))
print("TF-IDF classifier:", type(tfidf_classifier))
print("TF-IDF classes:", tfidf_classifier.classes_)

TF-IDF vectorizer: <class 'sklearn.feature_extraction.text.TfidfVectorizer'>
TF-IDF classifier: <class 'sklearn.svm._classes.LinearSVC'>
TF-IDF classes: ['conflict' 'negative' 'neutral' 'positive']


In [37]:
test_evaluation = test_split.copy()

test_evaluation["tfidf_input"] = (
    test_evaluation["aspect_term"]
    + " [SEP] "
    + test_evaluation["text"]
)

X_test_tfidf = tfidf_vectorizer.transform(
    test_evaluation["tfidf_input"]
)

test_evaluation["tfidf_prediction"] = (
    tfidf_classifier.predict(X_test_tfidf)
)

y_true = test_evaluation["polarity"]
y_tfidf = test_evaluation["tfidf_prediction"]

tfidf_accuracy = accuracy_score(
    y_true,
    y_tfidf
)

tfidf_macro_f1 = f1_score(
    y_true,
    y_tfidf,
    labels=labels,
    average="macro",
    zero_division=0
)

tfidf_weighted_f1 = f1_score(
    y_true,
    y_tfidf,
    labels=labels,
    average="weighted",
    zero_division=0
)

print("=" * 70)
print("TF-IDF — TRUE HELD-OUT TEST RESULTS")
print("=" * 70)

print("Accuracy:", round(tfidf_accuracy, 4))
print("Macro-F1:", round(tfidf_macro_f1, 4))
print("Weighted F1:", round(tfidf_weighted_f1, 4))

print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_tfidf,
        labels=labels,
        zero_division=0
    )
)

print("Confusion matrix:")
print(
    confusion_matrix(
        y_true,
        y_tfidf,
        labels=labels
    )
)

TF-IDF — TRUE HELD-OUT TEST RESULTS
Accuracy: 0.7326
Macro-F1: 0.5281
Weighted F1: 0.7305

Classification report:
              precision    recall  f1-score   support

    negative       0.72      0.77      0.75       102
     neutral       0.54      0.57      0.56        47
    positive       0.84      0.78      0.81       120
    conflict       0.00      0.00      0.00         4

    accuracy                           0.73       273
   macro avg       0.52      0.53      0.53       273
weighted avg       0.73      0.73      0.73       273

Confusion matrix:
[[79 12 10  1]
 [13 27  7  0]
 [15 11 94  0]
 [ 3  0  1  0]]


In [38]:
PAD_ID = 0
UNK_ID = 1


def tokenize(text):
    text = str(text).lower()

    return re.findall(
        r"[a-z0-9]+(?:'[a-z]+)?",
        text
    )


def encode_tokens(
    tokens,
    vocabulary,
    max_length
):
    token_ids = [
        vocabulary.get(token, UNK_ID)
        for token in tokens[:max_length]
    ]

    padding_needed = max_length - len(token_ids)

    if padding_needed > 0:
        token_ids.extend(
            [PAD_ID] * padding_needed
        )

    return token_ids

In [41]:
class ATAELSTM(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_classes,
        embedding_weights=None,
        dropout=0.30
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=PAD_ID
        )

        if embedding_weights is not None:
            self.embedding.weight.data.copy_(
                embedding_weights
            )

        self.lstm = nn.LSTM(
            input_size=embedding_dim * 2,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        lstm_output_dim = hidden_dim * 2

        self.attention_projection = nn.Linear(
            lstm_output_dim + embedding_dim,
            lstm_output_dim
        )

        self.attention_score = nn.Linear(
            lstm_output_dim,
            1,
            bias=False
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            lstm_output_dim,
            num_classes
        )

    def forward(
        self,
        sentence_ids,
        aspect_ids
    ):
        sentence_mask = sentence_ids.ne(PAD_ID)

        sentence_embeddings = self.embedding(
            sentence_ids
        )

        aspect_embeddings = self.embedding(
            aspect_ids
        )

        aspect_mask = aspect_ids.ne(
            PAD_ID
        ).unsqueeze(-1)

        aspect_sum = (
            aspect_embeddings * aspect_mask
        ).sum(dim=1)

        aspect_length = aspect_mask.sum(
            dim=1
        ).clamp(min=1)

        aspect_vector = aspect_sum / aspect_length

        expanded_aspect = aspect_vector.unsqueeze(
            1
        ).expand(
            -1,
            sentence_embeddings.size(1),
            -1
        )

        lstm_input = torch.cat(
            [
                sentence_embeddings,
                expanded_aspect
            ],
            dim=-1
        )

        lstm_output, _ = self.lstm(
            lstm_input
        )

        attention_input = torch.cat(
            [
                lstm_output,
                expanded_aspect
            ],
            dim=-1
        )

        attention_hidden = torch.tanh(
            self.attention_projection(
                attention_input
            )
        )

        attention_scores = self.attention_score(
            attention_hidden
        ).squeeze(-1)

        attention_scores = attention_scores.masked_fill(
            ~sentence_mask,
            -1e9
        )

        attention_weights = torch.softmax(
            attention_scores,
            dim=1
        )

        sentence_representation = torch.bmm(
            attention_weights.unsqueeze(1),
            lstm_output
        ).squeeze(1)

        sentence_representation = self.dropout(
            sentence_representation
        )

        logits = self.classifier(
            sentence_representation
        )

        return logits, attention_weights

In [42]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("ATAE-LSTM inference device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ATAE-LSTM inference device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [43]:
atae_checkpoint = torch.load(
    ATAE_MODEL_PATH,
    map_location=device
)

vocab = atae_checkpoint["vocab"]
label_to_id = atae_checkpoint["label_to_id"]
config = atae_checkpoint["config"]

id_to_label = {
    label_id: label
    for label, label_id in label_to_id.items()
}

atae_model = ATAELSTM(
    vocab_size=len(vocab),
    embedding_dim=config["embedding_dim"],
    hidden_dim=config["hidden_dim"],
    num_classes=len(label_to_id),
    dropout=config["dropout"]
).to(device)

atae_model.load_state_dict(
    atae_checkpoint["model_state_dict"]
)

atae_model.eval()

MAX_SENTENCE_LEN = config["max_sentence_len"]
MAX_ASPECT_LEN = config["max_aspect_len"]

print("ATAE-LSTM checkpoint loaded.")
print("Vocabulary size:", len(vocab))
print("Label mapping:", label_to_id)
print("Model config:", config)

if "best_validation_macro_f1" in atae_checkpoint:
    print(
        "Checkpoint best validation Macro-F1:",
        round(
            atae_checkpoint[
                "best_validation_macro_f1"
            ],
            4
        )
    )

ATAE-LSTM checkpoint loaded.
Vocabulary size: 1883
Label mapping: {'negative': 0, 'neutral': 1, 'positive': 2, 'conflict': 3}
Model config: {'embedding_dim': 100, 'hidden_dim': 128, 'dropout': 0.3, 'max_sentence_len': 100, 'max_aspect_len': 10}
Checkpoint best validation Macro-F1: 0.4842


In [44]:
def predict_atae_polarity(
    text,
    aspect_term,
    model,
    vocabulary
):
    model.eval()

    text_tokens = tokenize(text)
    aspect_tokens = tokenize(aspect_term)

    sentence_ids = encode_tokens(
        tokens=text_tokens,
        vocabulary=vocabulary,
        max_length=MAX_SENTENCE_LEN
    )

    aspect_ids = encode_tokens(
        tokens=aspect_tokens,
        vocabulary=vocabulary,
        max_length=MAX_ASPECT_LEN
    )

    sentence_tensor = torch.tensor(
        [sentence_ids],
        dtype=torch.long
    ).to(device)

    aspect_tensor = torch.tensor(
        [aspect_ids],
        dtype=torch.long
    ).to(device)

    with torch.no_grad():
        logits, _ = model(
            sentence_tensor,
            aspect_tensor
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )[0]

        predicted_label_id = torch.argmax(
            probabilities
        ).item()

    return {
        "predicted_polarity": id_to_label[
            predicted_label_id
        ],
        "confidence": float(
            probabilities[
                predicted_label_id
            ].item()
        )
    }

In [47]:
atae_predictions = []
atae_confidences = []

for row_number, (_, row) in enumerate(
    test_evaluation.iterrows(),
    start=1
):
    result = predict_atae_polarity(
        text=row["text"],
        aspect_term=row["aspect_term"],
        model=atae_model,
        vocabulary=vocab
    )

    atae_predictions.append(
        result["predicted_polarity"]
    )

    atae_confidences.append(
        result["confidence"]
    )

    if row_number % 50 == 0:
        print(
            f"Processed {row_number} "
            f"of {len(test_evaluation)} held-out test rows"
        )

test_evaluation["atae_prediction"] = atae_predictions
test_evaluation["atae_confidence"] = atae_confidences

y_atae = test_evaluation["atae_prediction"]

atae_accuracy = accuracy_score(
    y_true,
    y_atae
)

atae_macro_f1 = f1_score(
    y_true,
    y_atae,
    labels=labels,
    average="macro",
    zero_division=0
)

atae_weighted_f1 = f1_score(
    y_true,
    y_atae,
    labels=labels,
    average="weighted",
    zero_division=0
)

print("=" * 70)
print("ATAE-LSTM — TRUE HELD-OUT TEST RESULTS")
print("=" * 70)

print("Accuracy:", round(atae_accuracy, 4))
print("Macro-F1:", round(atae_macro_f1, 4))
print("Weighted F1:", round(atae_weighted_f1, 4))

print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_atae,
        labels=labels,
        zero_division=0
    )
)

print("Confusion matrix:")
print(
    confusion_matrix(
        y_true,
        y_atae,
        labels=labels
    )
)

Processed 50 of 273 held-out test rows
Processed 100 of 273 held-out test rows
Processed 150 of 273 held-out test rows
Processed 200 of 273 held-out test rows
Processed 250 of 273 held-out test rows
ATAE-LSTM — TRUE HELD-OUT TEST RESULTS
Accuracy: 0.7582
Macro-F1: 0.68
Weighted F1: 0.7657

Classification report:
              precision    recall  f1-score   support

    negative       0.74      0.77      0.76       102
     neutral       0.65      0.70      0.67        47
    positive       0.89      0.76      0.82       120
    conflict       0.31      1.00      0.47         4

    accuracy                           0.76       273
   macro avg       0.65      0.81      0.68       273
weighted avg       0.78      0.76      0.77       273

Confusion matrix:
[[79 10  7  6]
 [ 9 33  4  1]
 [19  8 91  2]
 [ 0  0  0  4]]


In [48]:
MODEL_EVALUATION_RESULTS = {
    "tfidf": {
        "test_accuracy": tfidf_accuracy,
        "test_macro_f1": tfidf_macro_f1,
        "test_weighted_f1": tfidf_weighted_f1
    },
    "atae_lstm": {
        "test_accuracy": atae_accuracy,
        "test_macro_f1": atae_macro_f1,
        "test_weighted_f1": atae_weighted_f1
    }
}

metrics_table = (
    pd.DataFrame(
        MODEL_EVALUATION_RESULTS
    )
    .T
    .reset_index()
    .rename(
        columns={
            "index": "model"
        }
    )
    .sort_values(
        "test_macro_f1",
        ascending=False
    )
    .reset_index(drop=True)
)

display(metrics_table)

,model,test_accuracy,test_macro_f1,test_weighted_f1
0,atae_lstm,0.758242,0.679965,0.765655
1,tfidf,0.732601,0.528082,0.730495


In [49]:
selected_model = metrics_table.iloc[0]["model"]

selected_macro_f1 = metrics_table.iloc[0][
    "test_macro_f1"
]

selected_accuracy = metrics_table.iloc[0][
    "test_accuracy"
]

print("=" * 70)
print("FINAL DASHBOARD MODEL SELECTION")
print("=" * 70)

print("Selected model:", selected_model)
print(
    "Reason: highest Macro-F1 on the shared "
    "held-out labeled test set."
)

print(
    "Selected Macro-F1:",
    round(selected_macro_f1, 4)
)

print(
    "Selected accuracy:",
    round(selected_accuracy, 4)
)

FINAL DASHBOARD MODEL SELECTION
Selected model: atae_lstm
Reason: highest Macro-F1 on the shared held-out labeled test set.
Selected Macro-F1: 0.68
Selected accuracy: 0.7582


In [50]:
held_out_output_columns = [
    "text",
    "aspect_term",
    "polarity",
    "tfidf_prediction",
    "atae_prediction",
    "atae_confidence"
]

held_out_test_predictions = test_evaluation[
    held_out_output_columns
].copy()

held_out_test_predictions.to_csv(
    EVALUATION_OUTPUT_PATH,
    index=False
)

metrics_table.to_csv(
    METRICS_OUTPUT_PATH,
    index=False
)

print("Saved held-out predictions to:")
print(EVALUATION_OUTPUT_PATH)

print("\nSaved evaluation metrics to:")
print(METRICS_OUTPUT_PATH)

Saved held-out predictions to:
c:\New folder\Projects\sentiment analysis\dataset\runtime\held_out_test_predictions.csv

Saved evaluation metrics to:
c:\New folder\Projects\sentiment analysis\dataset\runtime\model_evaluation_metrics.csv


In [51]:
runtime_required_paths = [
    TFIDF_PREDICTIONS_PATH,
    ATAE_PREDICTIONS_PATH
]

missing_runtime_paths = [
    path
    for path in runtime_required_paths
    if not path.exists()
]

if missing_runtime_paths:
    print(
        "Runtime comparison skipped because these "
        "prediction files are missing:"
    )

    for path in missing_runtime_paths:
        print("-", path)

    print(
        "\nRun these notebooks first if you want "
        "runtime agreement analysis:"
    )

    print("1. 03_extract_aspects.ipynb")
    print("2. 04_predict_tfidf.ipynb")
    print("3. 05_predict_atae_lstm.ipynb")

else:
    tfidf_runtime = pd.read_csv(
        TFIDF_PREDICTIONS_PATH
    )

    atae_runtime = pd.read_csv(
        ATAE_PREDICTIONS_PATH
    )

    print(
        "TF-IDF runtime prediction rows:",
        len(tfidf_runtime)
    )

    print(
        "ATAE-LSTM runtime prediction rows:",
        len(atae_runtime)
    )

TF-IDF runtime prediction rows: 11
ATAE-LSTM runtime prediction rows: 11


In [52]:
join_columns = [
    "comment_id",
    "text",
    "aspect_term"
]

required_runtime_columns = set(
    join_columns + [
        "predicted_polarity",
        "confidence"
    ]
)

missing_tfidf_columns = (
    required_runtime_columns
    - set(tfidf_runtime.columns)
)

missing_atae_columns = (
    required_runtime_columns
    - set(atae_runtime.columns)
)

if missing_tfidf_columns:
    raise ValueError(
        "TF-IDF runtime file missing: "
        + ", ".join(
            sorted(missing_tfidf_columns)
        )
    )

if missing_atae_columns:
    raise ValueError(
        "ATAE-LSTM runtime file missing: "
        + ", ".join(
            sorted(missing_atae_columns)
        )
    )

In [53]:
tfidf_runtime = tfidf_runtime.drop_duplicates(
    subset=[
        "comment_id",
        "aspect_term"
    ],
    keep="first"
).copy()

atae_runtime = atae_runtime.drop_duplicates(
    subset=[
        "comment_id",
        "aspect_term"
    ],
    keep="first"
).copy()

tfidf_for_merge = tfidf_runtime[
    [
        "comment_id",
        "text",
        "aspect_term",
        "predicted_polarity",
        "confidence"
    ]
].rename(
    columns={
        "predicted_polarity": "tfidf_polarity",
        "confidence": "tfidf_confidence"
    }
)

atae_for_merge = atae_runtime[
    [
        "comment_id",
        "text",
        "aspect_term",
        "predicted_polarity",
        "confidence"
    ]
].rename(
    columns={
        "predicted_polarity": "atae_polarity",
        "confidence": "atae_confidence"
    }
)

runtime_comparison = tfidf_for_merge.merge(
    atae_for_merge,
    on=join_columns,
    how="outer",
    indicator=True
)

print(
    "Rows by merge status:"
)

print(
    runtime_comparison["_merge"].value_counts()
)

display(runtime_comparison.head(20))

Rows by merge status:
_merge
both          11
left_only      0
right_only     0
Name: count, dtype: int64


,comment_id,text,aspect_term,tfidf_polarity,tfidf_confidence,atae_polarity,atae_confidence,_merge
0,c001,The laptop is excellent for school work and br...,8gb ram,negative,0.536259,negative,0.6688,both
1,c001,The laptop is excellent for school work and br...,productivity,positive,0.638705,positive,0.7052,both
2,c001,The laptop is excellent for school work and br...,school,positive,0.480804,positive,0.7729,both
3,c002,"Light games and older titles run fine, but AAA...",gaming performance,negative,0.510392,negative,0.9803,both
4,c003,"The touchscreen is useful, and Microsoft Offic...",office work,positive,0.532008,positive,0.6671,both
5,c003,"The touchscreen is useful, and Microsoft Offic...",productivity,positive,0.566816,positive,0.5065,both
6,c003,"The touchscreen is useful, and Microsoft Offic...",touchscreen,positive,0.421551,negative,0.4912,both
7,c004,"The laptop has poor gaming performance, althou...",gaming performance,negative,0.711015,negative,0.9609,both
8,c004,"The laptop has poor gaming performance, althou...",productivity,negative,0.357693,positive,0.7179,both
9,c005,"The battery life is decent, but storage fills ...",battery,negative,0.527033,negative,0.5114,both


In [54]:
shared_predictions = runtime_comparison[
    runtime_comparison["_merge"] == "both"
].copy()

shared_predictions["agreement"] = (
    shared_predictions["tfidf_polarity"]
    == shared_predictions["atae_polarity"]
)

shared_predictions["confidence_difference"] = (
    shared_predictions["atae_confidence"]
    - shared_predictions["tfidf_confidence"]
)

agreement_percentage = (
    shared_predictions["agreement"].mean() * 100
)

print(
    "Shared runtime comment-aspect pairs:",
    len(shared_predictions)
)

print(
    "TF-IDF / ATAE-LSTM agreement:",
    round(agreement_percentage, 2),
    "%"
)

Shared runtime comment-aspect pairs: 11
TF-IDF / ATAE-LSTM agreement: 81.82 %


In [55]:
runtime_disagreements = shared_predictions[
    ~shared_predictions["agreement"]
].copy()

runtime_disagreements = runtime_disagreements.sort_values(
    by="atae_confidence",
    ascending=False
)

print(
    "Runtime disagreements:",
    len(runtime_disagreements)
)

display(
    runtime_disagreements[
        [
            "comment_id",
            "text",
            "aspect_term",
            "tfidf_polarity",
            "tfidf_confidence",
            "atae_polarity",
            "atae_confidence",
            "confidence_difference"
        ]
    ].head(30)
)

Runtime disagreements: 2


,comment_id,text,aspect_term,tfidf_polarity,tfidf_confidence,atae_polarity,atae_confidence,confidence_difference
8,c004,"The laptop has poor gaming performance, althou...",productivity,negative,0.357693,positive,0.7179,0.360207
6,c003,"The touchscreen is useful, and Microsoft Offic...",touchscreen,positive,0.421551,negative,0.4912,0.069649


In [56]:
comparison_columns = [
    "comment_id",
    "text",
    "aspect_term",
    "tfidf_polarity",
    "tfidf_confidence",
    "atae_polarity",
    "atae_confidence",
    "agreement",
    "confidence_difference"
]

runtime_comparison_output = shared_predictions[
    comparison_columns
].copy()

runtime_comparison_output.to_csv(
    COMPARISON_OUTPUT_PATH,
    index=False
)

print("Saved runtime comparison:")
print(COMPARISON_OUTPUT_PATH)

display(runtime_comparison_output.head(20))

Saved runtime comparison:
c:\New folder\Projects\sentiment analysis\dataset\runtime\model_comparison.csv


,comment_id,text,aspect_term,tfidf_polarity,tfidf_confidence,atae_polarity,atae_confidence,agreement,confidence_difference
0,c001,The laptop is excellent for school work and br...,8gb ram,negative,0.536259,negative,0.6688,True,0.132541
1,c001,The laptop is excellent for school work and br...,productivity,positive,0.638705,positive,0.7052,True,0.066495
2,c001,The laptop is excellent for school work and br...,school,positive,0.480804,positive,0.7729,True,0.292096
3,c002,"Light games and older titles run fine, but AAA...",gaming performance,negative,0.510392,negative,0.9803,True,0.469908
4,c003,"The touchscreen is useful, and Microsoft Offic...",office work,positive,0.532008,positive,0.6671,True,0.135092
5,c003,"The touchscreen is useful, and Microsoft Offic...",productivity,positive,0.566816,positive,0.5065,True,-0.060316
6,c003,"The touchscreen is useful, and Microsoft Offic...",touchscreen,positive,0.421551,negative,0.4912,False,0.069649
7,c004,"The laptop has poor gaming performance, althou...",gaming performance,negative,0.711015,negative,0.9609,True,0.249885
8,c004,"The laptop has poor gaming performance, althou...",productivity,negative,0.357693,positive,0.7179,False,0.360207
9,c005,"The battery life is decent, but storage fills ...",battery,negative,0.527033,negative,0.5114,True,-0.015633
